# 05. Thí nghiệm rò rỉ dữ liệu

**Rò rỉ dữ liệu** là khi thông tin của tương lai lọt vào quá trình huấn luyện hoặc đánh giá, làm sai số trông đẹp hơn thực tế ("đẹp giả").

**Thí nghiệm theo đề cương:** chuẩn hóa (z-score) chuỗi bằng trung bình và độ lệch chuẩn rồi mới dự báo.
- Cách **đúng**: tại mỗi mốc, chỉ dùng thống kê của dữ liệu *trước* mốc đó.
- Cách **sai (rò rỉ)**: dùng thống kê của *toàn chuỗi*, tức đã nhìn trước cả tương lai.

Ta so hai con số: **MAE chuẩn hóa** (sai số theo đơn vị z, thứ hay được báo cáo sau khi chuẩn hóa) và **MAE gốc** (sai số thật, đã đổi về đơn vị ban đầu). Hàm nằm trong `src/cpi_forecast/evaluation/leakage.py`.

In [1]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Kết quả (tầm 1 tháng, cả hai chuỗi)

In [2]:
from cpi_forecast.evaluation.leakage import rolling_origin_zscore

df = pd.read_csv("../data/processed/cpi_monthly_clean.csv", parse_dates=["date"])
from cpi_forecast.models.naive import NaiveForecaster
from cpi_forecast.models.seasonal_naive import SeasonalNaiveForecaster
from cpi_forecast.models.moving_average import MovingAverageForecaster
from cpi_forecast.models.trend_ols import TrendOLSForecaster

models = {
    "Naive": NaiveForecaster,
    "Seasonal Naive": SeasonalNaiveForecaster,
    "Trung bình trượt (k=3)": lambda: MovingAverageForecaster(k=3),
    "Hồi quy xu hướng (OLS)": TrendOLSForecaster,
}

rows = []
for col in ["index", "pct"]:
    y = df[col].to_numpy()
    for name, factory in models.items():
        z_ok, g_ok = rolling_origin_zscore(y, factory, h=1, leaky=False)
        z_lk, g_lk = rolling_origin_zscore(y, factory, h=1, leaky=True)
        rows.append({"Chuỗi": col, "Mô hình": name,
                     "MAE chuẩn hóa (đúng)": round(z_ok, 3),
                     "MAE chuẩn hóa (rò rỉ)": round(z_lk, 3),
                     "Tỉ lệ rò rỉ/đúng": round(z_lk / z_ok, 2),
                     "MAE gốc (đúng)": round(g_ok, 3),
                     "MAE gốc (rò rỉ)": round(g_lk, 3)})
print(pd.DataFrame(rows).to_string(index=False))

Chuỗi                Mô hình  MAE chuẩn hóa (đúng)  MAE chuẩn hóa (rò rỉ)  Tỉ lệ rò rỉ/đúng  MAE gốc (đúng)  MAE gốc (rò rỉ)
index                  Naive                 0.026                  0.017              0.66           1.114            1.114
index         Seasonal Naive                 0.255                  0.163              0.64          10.493           10.493
index Trung bình trượt (k=3)                 0.048                  0.031              0.65           1.985            1.985
index Hồi quy xu hướng (OLS)                 0.283                  0.215              0.76          13.791           13.791
  pct                  Naive                 0.452                  0.522              1.15           0.375            0.375
  pct         Seasonal Naive                 0.609                  0.719              1.18           0.517            0.517
  pct Trung bình trượt (k=3)                 0.486                  0.565              1.16           0.406            0.406


## Cách đọc

1. **MAE gốc giống hệt nhau** giữa hai cách. Lý do: cả 4 mô hình không đổi dự báo khi chuẩn hóa (chuẩn hóa chỉ đổi đơn vị đo). Vậy rò rỉ qua chuẩn hóa **không làm dự báo tốt lên thật**.
2. **MAE chuẩn hóa thì khác nhau.** Trên chuỗi chỉ số, cách rò rỉ cho con số nhỏ hơn 24 đến 36% (tỉ lệ 0,64 đến 0,76): sai số trông "đẹp" hơn thực tế. Lý do khả dĩ: độ lệch chuẩn của toàn chuỗi (khoảng 64) lớn hơn độ lệch chuẩn của phần quá khứ, vì chỉ số tăng nên độ phân tán lớn dần; chia cho số lớn hơn thì sai số "nhỏ lại".
3. Trên chuỗi % thay đổi thì **ngược lại**: cách rò rỉ cho con số lớn hơn 15 đến 18% (tỉ lệ 1,15 đến 1,18).

**Kết luận:** rò rỉ làm con số sai số báo cáo bị sai lệch, và chiều sai lệch phụ thuộc vào dữ liệu nên không thể tin vào nó. Cải thiện ở chuỗi chỉ số nằm ở "cây thước", không phải ở mô hình.